# Iteration 8 — RQ2  

This notebook combines two roles in a **single file** while keeping them methodologically separated:

### Part A — Exploratory phase
Used for:
- task understanding
- prompt design
- few-shot exemplar construction
- smoke testing
- debugging
- validation-only error analysis

### Part B — Frozen benchmark phase
Used for:
- final held-out test evaluation
- fair model comparison
- RQ2 reporting

## Important rule
Only the **benchmark section** counts as the final result for thesis reporting.  
The exploratory section is for development only and must not leak test information.

## Task
Binary classification of incident reports into:

- `Process Safety`
- `Non-Process Safety`

## Hardware
This notebook is designed for **MacBook Air M4 / Apple Silicon** and therefore uses an **MPS-safe** workflow.


In [ ]:
# =============================================================================
# IMPORTS, WARNINGS, RANDOMNESS
# =============================================================================
import os
import re
import gc
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import plotly.express as px
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"


## Runtime Configuration

`RUN_XL = False` is the safe default for Apple Silicon laptops.  
Enable `RUN_XL = True` only if your machine can load `flan-t5-xl` reliably.


In [ ]:
# =============================================================================
# DEVICE + MODEL CONFIGURATION
# =============================================================================
DEVICE = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print("Using device:", DEVICE)

DEBUG_MODE = True
RUN_XL = False

if DEBUG_MODE:
    MAX_SOURCE_TOKENS = 320
else:
    MAX_SOURCE_TOKENS = 384

MAX_LABEL_TOKENS = 16
CHECKPOINT_EVERY = 50
RQ2_TARGET_MACRO_F1 = 0.7825

MODELS_TO_COMPARE = ["google/flan-t5-large"]
if RUN_XL:
    MODELS_TO_COMPARE.append("google/flan-t5-xl")

CLASS_LABELS = ["Process Safety", "Non-Process Safety"]

print("Models:", MODELS_TO_COMPARE)
print("MAX_SOURCE_TOKENS:", MAX_SOURCE_TOKENS)
print("RQ2 target macro-F1:", RQ2_TARGET_MACRO_F1)


## Project Root Detection and Paths


In [ ]:
# =============================================================================
# PROJECT ROOT + PATHS
# =============================================================================
def find_project_root_with_datasets(start_path: Path, max_levels: int = 10) -> Path:
    cur = start_path.resolve()
    for _ in range(max_levels):
        if (cur / "Datasets").exists() or (cur / "Master Dataset 34k").exists():
            return cur
        cur = cur.parent
    raise FileNotFoundError(
        "Could not find project root with 'Datasets' or 'Master Dataset 34k'. "
        "Set THESIS_BASE_DIR if needed."
    )

env_base = os.environ.get("THESIS_BASE_DIR")
if env_base:
    BASE_DIR = Path(env_base).resolve()
else:
    BASE_DIR = find_project_root_with_datasets(Path.cwd())

PATHS = {
    "base": BASE_DIR,
    "master_dataset": BASE_DIR / "Master Dataset 34k",
    "results": BASE_DIR / "Results" / "_iteration_8_merged",
    "artifacts": BASE_DIR / "Results" / "_iteration_8_merged" / "artifacts",
    "metrics": BASE_DIR / "Results" / "_iteration_8_merged" / "metrics",
    "figures": BASE_DIR / "Results" / "_iteration_8_merged" / "figures",
}

for key, path in PATHS.items():
    if key not in {"base", "master_dataset"}:
        path.mkdir(parents=True, exist_ok=True)

DATASET_FILES = {
    "english_manual":  PATHS["master_dataset"] / "By_SL_Country" / "master_df_English_manual.json",
    "german_manual":   PATHS["master_dataset"] / "By_SL_Country" / "master_df_Germany_manual.json",
    "swedish_manual":  PATHS["master_dataset"] / "By_SL_Country" / "master_df_Sweden_manual.json",
    "dutch_manual":    PATHS["master_dataset"] / "By_SL_Country" / "master_df_Netherlands_manual.json",
}

DATASET_FILES = {k: v for k, v in DATASET_FILES.items() if v.exists()}

DATASET_KEY_TO_LANGUAGE = {
    "english_manual": "English",
    "german_manual": "German",
    "swedish_manual": "Swedish",
    "dutch_manual": "Dutch",
}

print("BASE_DIR:", BASE_DIR)
print("Available datasets:", list(DATASET_FILES.keys()))


## Load and Combine Multilingual Data


In [ ]:
# =============================================================================
# LOAD AND COMBINE DATA
# =============================================================================
all_dfs = []

for dataset_key, json_path in DATASET_FILES.items():
    language = DATASET_KEY_TO_LANGUAGE.get(dataset_key, "Unknown")
    df_part = pd.read_json(json_path)

    required_cols = ["TITLE", "CASE_DESCRIPTION", "CASE_TYPE"]
    missing = [c for c in required_cols if c not in df_part.columns]
    if missing:
        raise ValueError(f"{json_path.name} is missing required columns: {missing}")

    keep_cols = [c for c in ["TITLE", "CASE_DESCRIPTION", "CASE_TYPE", "CASENO", "SL_COUNTRY"] if c in df_part.columns]
    df_part = df_part[keep_cols].copy()
    df_part["LANGUAGE"] = language
    df_part["DATASET_KEY"] = dataset_key

    all_dfs.append(df_part)

if not all_dfs:
    raise FileNotFoundError("No dataset JSON files were found.")

df = pd.concat(all_dfs, ignore_index=True)

print("Combined shape:", df.shape)
df.head()


## Clean Text and Create the Binary Target


In [ ]:
# =============================================================================
# CLEANING + TARGET
# =============================================================================
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

for col in ["TITLE", "CASE_DESCRIPTION", "CASE_TYPE", "LANGUAGE"]:
    df[col] = df[col].apply(clean_text)

df["TEXT"] = (
    "Title: " + df["TITLE"].fillna("") +
    "\nDescription: " + df["CASE_DESCRIPTION"].fillna("")
).str.strip()

df = df[df["TEXT"].str.len() > 20].copy()

df["binary_label"] = np.where(
    df["CASE_TYPE"] == "Process Safety",
    "Process Safety",
    "Non-Process Safety"
)

df["text_id"] = pd.factorize(df["TEXT"])[0]

print(df["binary_label"].value_counts())
print("Rows after cleaning:", len(df))


## Build a Leakage-Safe Grouped Split

This split is created **before** prompt design so that:
- prompt design uses only train/validation data
- the final test set remains untouched until the benchmark section


In [ ]:
# =============================================================================
# SPLIT BY UNIQUE TEXTS
# =============================================================================
unique_for_split = (
    df[["text_id", "TEXT", "binary_label", "LANGUAGE"]]
    .drop_duplicates("text_id")
    .reset_index(drop=True)
)

min_class_count = unique_for_split["binary_label"].value_counts().min()
if min_class_count < 2:
    raise ValueError("Not enough examples per class to create a grouped split.")

n_splits = min(5, int(min_class_count))
sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
splits = list(
    sgkf.split(
        X=unique_for_split["TEXT"],
        y=unique_for_split["binary_label"],
        groups=unique_for_split["text_id"]
    )
)

train_idx, test_idx = splits[0]
train_val = unique_for_split.iloc[train_idx].copy()
test_unique = unique_for_split.iloc[test_idx].copy()

min_class_count_inner = train_val["binary_label"].value_counts().min()
n_splits_inner = min(5, int(min_class_count_inner)) if min_class_count_inner >= 2 else 2

sgkf_inner = StratifiedGroupKFold(n_splits=n_splits_inner, shuffle=True, random_state=SEED)
inner_splits = list(
    sgkf_inner.split(
        X=train_val["TEXT"],
        y=train_val["binary_label"],
        groups=train_val["text_id"]
    )
)

inner_train_idx, inner_val_idx = inner_splits[0]
train_unique = train_val.iloc[inner_train_idx].copy()
val_unique = train_val.iloc[inner_val_idx].copy()

split_assignments = pd.concat([
    train_unique.assign(split="train"),
    val_unique.assign(split="val"),
    test_unique.assign(split="test")
], ignore_index=True)

split_assignments.to_csv(PATHS["artifacts"] / "split_assignments.csv", index=False)
split_assignments["split"].value_counts()


# Part A — Exploratory Phase (Train/Validation Only)

The cells below are used for:
- solving the task
- inspecting the data
- designing the prompt
- debugging model behaviour
- performing validation-only error analysis

**Do not use the test set in this phase.**


## Exploratory Data Overview


In [ ]:
# =============================================================================
# EXPLORATORY DATA OVERVIEW
# =============================================================================
train_val_rows = df.merge(split_assignments[["text_id", "split"]], on="text_id", how="left")
train_val_rows = train_val_rows[train_val_rows["split"].isin(["train", "val"])].copy()

overview = (
    train_val_rows.groupby(["split", "LANGUAGE", "binary_label"])
    .size()
    .reset_index(name="count")
    .sort_values(["split", "LANGUAGE", "binary_label"])
)

overview.to_csv(PATHS["artifacts"] / "train_val_label_overview.csv", index=False)
overview.head(20)


## Build Train-Only Few-Shot Exemplars

Few-shot examples are selected only from the training split.


In [ ]:
# =============================================================================
# FEW-SHOT EXEMPLARS FROM TRAIN ONLY
# =============================================================================
train_rows = df.merge(split_assignments[["text_id", "split"]], on="text_id", how="left")
train_rows = train_rows[train_rows["split"] == "train"].copy()

EXEMPLARS = {}

for language in sorted(train_rows["LANGUAGE"].unique()):
    EXEMPLARS[language] = {}
    for label in CLASS_LABELS:
        subset = train_rows[
            (train_rows["LANGUAGE"] == language) &
            (train_rows["binary_label"] == label)
        ].drop_duplicates("text_id")

        sample_n = min(2, len(subset))
        sampled = subset.sample(sample_n, random_state=SEED) if sample_n > 0 else subset

        EXEMPLARS[language][label] = [
            {
                "title": row["TITLE"],
                "description": row["CASE_DESCRIPTION"]
            }
            for _, row in sampled.iterrows()
        ]

EXEMPLARS["DEFAULT"] = EXEMPLARS.get("English", EXEMPLARS[next(iter(EXEMPLARS))])

with open(PATHS["artifacts"] / "few_shot_exemplars.json", "w", encoding="utf-8") as f:
    json.dump(EXEMPLARS, f, indent=2)

EXEMPLARS


## Prompt Definition

This prompt is used both in validation debugging and in the final benchmark.  
If you change it, do so **before** running the benchmark section.


In [ ]:
# =============================================================================
# PROMPT DEFINITION + SCORING
# =============================================================================
PROCESS_SAFETY_DEFINITION = '''
A Process Safety incident involves failures, leaks, releases, trips, shutdowns, pressure deviations,
fires, explosions, ruptures, or other hazardous events arising from process equipment, process systems,
or industrial plant operations.

A Non-Process Safety incident includes personal safety incidents, slips, trips, falls, minor administrative
issues, office/workshop events, traffic/vehicle incidents, or other events not arising from process systems.
'''.strip()

def get_language_exemplars(language: str):
    if language in EXEMPLARS:
        return EXEMPLARS[language]
    return EXEMPLARS["DEFAULT"]

def build_few_shot_prompt(title: str, description: str, language: str) -> str:
    ex = get_language_exemplars(language)

    blocks = []
    for label in CLASS_LABELS:
        for item in ex.get(label, []):
            blocks.append(
                f"Example\n"
                f"Title: {item['title']}\n"
                f"Description: {item['description']}\n"
                f"Classification: {label}"
            )

    examples_text = "\n\n".join(blocks)

    prompt = (
        "You are an expert industrial safety analyst.\n"
        "Classify the incident into exactly one of the following labels:\n"
        "- Process Safety\n"
        "- Non-Process Safety\n\n"
        f"{PROCESS_SAFETY_DEFINITION}\n\n"
        f"{examples_text}\n\n"
        f"Now classify this incident.\n"
        f"Title: {title}\n"
        f"Description: {description}\n\n"
        "Answer:"
    )
    return prompt

def build_label_token_cache(tokenizer):
    cache = {}
    for label in CLASS_LABELS:
        cache[label] = tokenizer(
            label,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LABEL_TOKENS
        ).input_ids
    return cache

@torch.no_grad()
def score_candidate_labels(prompt: str, tokenizer, model, label_token_cache):
    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SOURCE_TOKENS
    )

    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    scores = []
    for label in CLASS_LABELS:
        label_ids = label_token_cache[label].to(DEVICE)

        out = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=label_ids
        )

        seq_len = label_ids.shape[1]
        sequence_log_score = -float(out.loss.detach().cpu()) * seq_len
        scores.append(sequence_log_score)

    scores = np.array(scores, dtype=np.float64)
    probs = np.exp(scores - scores.max())
    probs = probs / probs.sum()

    scored = pd.DataFrame({
        "label": CLASS_LABELS,
        "score": scores,
        "prob": probs
    }).sort_values("score", ascending=False).reset_index(drop=True)

    return scored

def classify_incident(title, description, language, tokenizer, model, label_token_cache):
    prompt = build_few_shot_prompt(title=title, description=description, language=language)
    scored = score_candidate_labels(prompt, tokenizer, model, label_token_cache)

    top1 = scored.iloc[0]
    top2 = scored.iloc[1]

    return {
        "pred_label": top1["label"],
        "pred_confidence": float(top1["prob"]),
        "score_margin": float(top1["score"] - top2["score"]),
        "top2_labels": scored["label"].tolist(),
        "top2_probs": scored["prob"].tolist(),
        "prompt": prompt
    }


## Prompt Inspection

Inspect a single prompt before running model inference.


In [ ]:
# =============================================================================
# PROMPT INSPECTION
# =============================================================================
sample_train = train_rows.drop_duplicates("text_id").iloc[0]
sample_prompt = build_few_shot_prompt(
    title=sample_train["TITLE"],
    description=sample_train["CASE_DESCRIPTION"],
    language=sample_train["LANGUAGE"]
)

print(sample_prompt[:4000])


## Smoke Test on a Single Validation Example


In [ ]:
# =============================================================================
# SINGLE-EXAMPLE SMOKE TEST
# =============================================================================
sample_val = df.merge(split_assignments[["text_id", "split"]], on="text_id", how="left")
sample_val = sample_val[sample_val["split"] == "val"].drop_duplicates("text_id").iloc[0]

tokenizer = AutoTokenizer.from_pretrained(MODELS_TO_COMPARE[0])
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODELS_TO_COMPARE[0],
    torch_dtype=torch.float32
).to(DEVICE)
model.eval()

label_token_cache = build_label_token_cache(tokenizer)

smoke = classify_incident(
    title=sample_val["TITLE"],
    description=sample_val["CASE_DESCRIPTION"],
    language=sample_val["LANGUAGE"],
    tokenizer=tokenizer,
    model=model,
    label_token_cache=label_token_cache
)
smoke


## Validation Debugging Run

This run is **not** the final benchmark.  
It is only for checking whether the prompt and exemplars behave sensibly on the validation split.


In [ ]:
# =============================================================================
# VALIDATION DEBUGGING RUN
# =============================================================================
def clear_memory():
    gc.collect()
    if torch.backends.mps.is_available():
        try:
            torch.mps.empty_cache()
        except Exception:
            pass

val_rows = df.merge(split_assignments[["text_id", "split"]], on="text_id", how="left")
val_rows = val_rows[val_rows["split"] == "val"].drop_duplicates("text_id").reset_index(drop=True)

# small subset for debugging
debug_val = val_rows.groupby("LANGUAGE", group_keys=False).head(5).reset_index(drop=True)

debug_predictions = []

for _, row in tqdm(debug_val.iterrows(), total=len(debug_val), desc="validation_debug"):
    result = classify_incident(
        title=row["TITLE"],
        description=row["CASE_DESCRIPTION"],
        language=row["LANGUAGE"],
        tokenizer=tokenizer,
        model=model,
        label_token_cache=label_token_cache
    )

    debug_predictions.append({
        "text_id": row["text_id"],
        "LANGUAGE": row["LANGUAGE"],
        "TITLE": row["TITLE"],
        "CASE_DESCRIPTION": row["CASE_DESCRIPTION"],
        "gold_label": row["binary_label"],
        "pred_label": result["pred_label"],
        "pred_confidence": result["pred_confidence"],
        "score_margin": result["score_margin"],
    })

    clear_memory()

debug_val_df = pd.DataFrame(debug_predictions)
debug_val_df.to_csv(PATHS["artifacts"] / "validation_debug_predictions.csv", index=False)
debug_val_df


## Validation Error Analysis

Use this section to inspect whether the prompt design is sensible before freezing the benchmark setup.


In [ ]:
# =============================================================================
# VALIDATION ERROR ANALYSIS
# =============================================================================
val_errors = debug_val_df[debug_val_df["gold_label"] != debug_val_df["pred_label"]].copy()
val_errors.to_csv(PATHS["artifacts"] / "validation_debug_errors.csv", index=False)
val_errors


# Freeze Point

From this point onward:

- do **not** change the split
- do **not** change the label set
- do **not** use test examples for prompt design
- do **not** modify few-shot examples using test feedback

Everything below is the **final benchmark section**.


# Part B — Final Fair Benchmark

This section runs the held-out evaluation on the **test split only**.


## Benchmark Function


In [ ]:
# =============================================================================
# BENCHMARK FUNCTION
# =============================================================================
def benchmark_model(model_name: str, df_full: pd.DataFrame, split_df: pd.DataFrame):
    safe_model_name = model_name.replace("/", "_").replace("-", "_")
    pred_file = PATHS["artifacts"] / f"predictions_{safe_model_name}.csv"
    checkpoint_file = PATHS["artifacts"] / f"checkpoint_{safe_model_name}.csv"

    eval_rows = df_full.merge(split_df[["text_id", "split"]], on="text_id", how="left")
    eval_rows = eval_rows[eval_rows["split"] == "test"].copy()
    eval_rows = eval_rows.drop_duplicates("text_id").reset_index(drop=True)

    print(f"Loading model: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32
    ).to(DEVICE)
    model.eval()

    label_token_cache = build_label_token_cache(tokenizer)

    if checkpoint_file.exists():
        done_df = pd.read_csv(checkpoint_file)
        done_ids = set(done_df["text_id"].tolist())
        print(f"Resuming {model_name}. Completed rows: {len(done_ids)}")
    else:
        done_df = pd.DataFrame()
        done_ids = set()

    remaining = eval_rows[~eval_rows["text_id"].isin(done_ids)].copy()
    print(f"Remaining rows for {model_name}: {len(remaining)}")

    new_rows = []

    for i, row in tqdm(remaining.iterrows(), total=len(remaining), desc=model_name.split("/")[-1]):
        result = classify_incident(
            title=row["TITLE"],
            description=row["CASE_DESCRIPTION"],
            language=row["LANGUAGE"],
            tokenizer=tokenizer,
            model=model,
            label_token_cache=label_token_cache
        )

        new_rows.append({
            "text_id": row["text_id"],
            "LANGUAGE": row["LANGUAGE"],
            "TITLE": row["TITLE"],
            "CASE_DESCRIPTION": row["CASE_DESCRIPTION"],
            "gold_label": row["binary_label"],
            "pred_label": result["pred_label"],
            "pred_confidence": result["pred_confidence"],
            "score_margin": result["score_margin"],
            "top2_labels": json.dumps(result["top2_labels"]),
            "top2_probs": json.dumps(result["top2_probs"]),
        })

        clear_memory()

        if (i + 1) % CHECKPOINT_EVERY == 0:
            checkpoint_df = pd.concat([done_df, pd.DataFrame(new_rows)], ignore_index=True)
            checkpoint_df.to_csv(checkpoint_file, index=False)

    final_pred = pd.concat([done_df, pd.DataFrame(new_rows)], ignore_index=True)
    final_pred.to_csv(pred_file, index=False)

    del model
    del tokenizer
    clear_memory()

    return final_pred, pred_file


## Run the Final Benchmark for All Selected Models


In [ ]:
# =============================================================================
# RUN BENCHMARK
# =============================================================================
all_model_metrics = {}
all_model_prediction_files = {}

for model_name in MODELS_TO_COMPARE:
    print("\n" + "=" * 80)
    print("RUNNING FINAL BENCHMARK:", model_name)
    print("=" * 80)

    pred_df, pred_path = benchmark_model(model_name, df, split_assignments)
    all_model_prediction_files[model_name] = str(pred_path)

    y_true = pred_df["gold_label"]
    y_pred = pred_df["pred_label"]

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "ps_precision": precision_score(y_true, y_pred, labels=["Process Safety"], average="macro", zero_division=0),
        "ps_recall": recall_score(y_true, y_pred, labels=["Process Safety"], average="macro", zero_division=0),
        "ps_f1": f1_score(y_true, y_pred, labels=["Process Safety"], average="macro", zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=CLASS_LABELS).tolist(),
        "classification_report": classification_report(
            y_true, y_pred, labels=CLASS_LABELS, output_dict=True, zero_division=0
        ),
    }

    all_model_metrics[model_name] = metrics

print("Completed benchmark models:", list(all_model_metrics.keys()))


## Save Metrics and Build the Comparison Table


In [ ]:
# =============================================================================
# SAVE METRICS + SUMMARY TABLE
# =============================================================================
summary_rows = []
for model_name, metrics in all_model_metrics.items():
    summary_rows.append({
        "Model": model_name.split("/")[-1],
        "Accuracy": metrics["accuracy"],
        "Macro Precision": metrics["macro_precision"],
        "Macro Recall": metrics["macro_recall"],
        "Macro F1": metrics["macro_f1"],
        "PS Precision": metrics["ps_precision"],
        "PS Recall": metrics["ps_recall"],
        "PS F1": metrics["ps_f1"],
        "RQ2 Target Met": metrics["macro_f1"] >= RQ2_TARGET_MACRO_F1,
        "Gap to Target": metrics["macro_f1"] - RQ2_TARGET_MACRO_F1,
        "Prediction File": all_model_prediction_files[model_name],
    })

summary_df = pd.DataFrame(summary_rows).sort_values("Macro F1", ascending=False)
summary_df.to_csv(PATHS["metrics"] / "model_comparison_summary.csv", index=False)

with open(PATHS["metrics"] / "all_model_metrics.json", "w", encoding="utf-8") as f:
    json.dump(all_model_metrics, f, indent=2)

summary_df


## Confusion Matrices


In [ ]:
# =============================================================================
# CONFUSION MATRICES
# =============================================================================
for model_name, metrics in all_model_metrics.items():
    cm = np.array(metrics["confusion_matrix"])
    short_name = model_name.split("/")[-1]

    cm_df = pd.DataFrame(cm, index=CLASS_LABELS, columns=CLASS_LABELS)

    fig = px.imshow(
        cm_df,
        text_auto=True,
        aspect="auto",
        title=f"Confusion Matrix — {short_name}"
    )
    fig.write_image(PATHS["figures"] / f"confusion_matrix_{short_name}.png", scale=4)


## Test Error Analysis

This error analysis is now based on the **held-out test set**, not the exploratory validation subset.


In [ ]:
# =============================================================================
# TEST ERROR ANALYSIS
# =============================================================================
# Use the best-performing model for the main error-analysis file
best_model_name = summary_df.iloc[0]["Model"]
best_model_full_name = [m for m in MODELS_TO_COMPARE if m.split("/")[-1] == best_model_name][0]

best_pred_path = Path(all_model_prediction_files[best_model_full_name])
best_pred_df = pd.read_csv(best_pred_path)

test_errors = best_pred_df[best_pred_df["gold_label"] != best_pred_df["pred_label"]].copy()
test_errors.to_csv(PATHS["artifacts"] / f"test_errors_{best_model_name}.csv", index=False)

test_errors.head(30)


## Performance Comparison Plot


In [ ]:
# =============================================================================
# PERFORMANCE COMPARISON PLOT
# =============================================================================
plot_df = summary_df.copy()
plot_df["Macro F1 (%)"] = plot_df["Macro F1"] * 100
plot_df["Accuracy (%)"] = plot_df["Accuracy"] * 100

fig = px.bar(
    plot_df.melt(
        id_vars="Model",
        value_vars=["Macro F1 (%)", "Accuracy (%)"],
        var_name="Metric",
        value_name="Value"
    ),
    x="Model",
    y="Value",
    color="Metric",
    barmode="group",
    title="Iteration 8 Merged — Final Benchmark Comparison"
)
fig.write_image(PATHS["figures"] / "model_comparison_bar.png", scale=4)

fig


## Save Final Iteration Summary


In [ ]:
# =============================================================================
# FINAL SUMMARY JSON
# =============================================================================
iteration_summary = {
    "iteration": 8,
    "research_question": "RQ2 - Merged exploratory + benchmark notebook",
    "task": "Binary Process Safety vs Non-Process Safety classification",
    "models": MODELS_TO_COMPARE,
    "languages": sorted(df["LANGUAGE"].unique().tolist()),
    "data_rows": int(len(df)),
    "unique_texts": int(df["text_id"].nunique()),
    "rq2_target_macro_f1": RQ2_TARGET_MACRO_F1,
    "exploratory_outputs": {
        "validation_debug_predictions": str(PATHS["artifacts"] / "validation_debug_predictions.csv"),
        "validation_debug_errors": str(PATHS["artifacts"] / "validation_debug_errors.csv"),
        "few_shot_exemplars": str(PATHS["artifacts"] / "few_shot_exemplars.json"),
    },
    "benchmark_outputs": {
        "summary_csv": str(PATHS["metrics"] / "model_comparison_summary.csv"),
        "metrics_json": str(PATHS["metrics"] / "all_model_metrics.json"),
    },
    "results": {}
}

for model_name, metrics in all_model_metrics.items():
    short_name = model_name.split("/")[-1]
    iteration_summary["results"][short_name] = {
        "accuracy": metrics["accuracy"],
        "macro_precision": metrics["macro_precision"],
        "macro_recall": metrics["macro_recall"],
        "macro_f1": metrics["macro_f1"],
        "target_met": metrics["macro_f1"] >= RQ2_TARGET_MACRO_F1,
        "gap_to_target": metrics["macro_f1"] - RQ2_TARGET_MACRO_F1,
        "prediction_file": all_model_prediction_files[model_name],
    }

with open(PATHS["results"] / "iteration_8_merged_summary.json", "w", encoding="utf-8") as f:
    json.dump(iteration_summary, f, indent=2)

iteration_summary


## Final Note

This merged notebook contains both:
- the **problem-solving / prompt-design phase**
- the **final benchmark phase**

For the thesis, report only the **benchmark section** as the final result.


In [ ]:
print("Iteration 8 merged notebook complete.")
print("Results directory:", PATHS["results"])
